# Beta-belief baseline probe

This notebook answers two questions:

1. How does an agent's private Beta-form belief determine its probability of producing a supportive ($+1$) or opposing ($-1$) message?
2. How does the agent's Beta belief change after it consumes a given number and stance composition of messages?

The model has no public "neutral stance": every externally supplied expression event produces either $-1$ or $+1$. A symmetric private belief indicates only a long-run $50/50$ expression propensity, not a third message category. Production reads only the current private belief; consumption follows a homogeneous Beta–Bernoulli baseline. Networks, platforms, opinion leaders, source weights, memory, abstention, and empirical calibration are outside this probe's scope.

**Decision status.** The Beta belief, posterior-mass production rule, and homogeneous consumption rule form a provisional baseline for mechanism inspection. Successful execution verifies the implementation only; it does not validate the psychological mechanism.


## 1. Private belief, public message, and summaries

An agent's private belief is

$$P_t\sim\operatorname{Beta}(a_t,b_t),\qquad P_t\in[0,1].$$

The corresponding signed latent coordinate is

$$\Theta_t=2P_t-1\in[-1,1].$$

From the same state, we derive the mean belief $\mu_t=a_t/(a_t+b_t)$, signed mean $\bar\theta_t=2\mu_t-1$, concentration $\kappa_t=a_t+b_t$, and posterior mass on the supportive side:

$$\pi_t^+=P(P_t>0.5)=1-F_{\mathrm{Beta}}(0.5;a_t,b_t).$$

Message production uses probability matching:

$$Y_t\sim\operatorname{Bernoulli}(\pi_t^+),\qquad M_t=2Y_t-1\in\{-1,+1\}.$$

The direction of the private belief therefore does not determine any single message; it changes only the generation probabilities of the two message stances.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from numbers import Integral

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import beta as beta_distribution


@dataclass(frozen=True)
class BetaBelief:
    """Private belief over support for the positive proposition."""

    a: float
    b: float

    def __post_init__(self) -> None:
        if not (np.isfinite(self.a) and np.isfinite(self.b)):
            raise ValueError("Beta shapes must be finite.")
        if self.a <= 0 or self.b <= 0:
            raise ValueError("Beta shapes must be strictly positive.")


def support_probability(state: BetaBelief) -> float:
    """Posterior mass on the supportive side of the private belief."""
    return float(beta_distribution.sf(0.5, state.a, state.b))


def message_probabilities(state: BetaBelief) -> dict[str, float]:
    p_support = support_probability(state)
    return {"support": p_support, "oppose": 1.0 - p_support}


def produce_message(state: BetaBelief, rng: np.random.Generator) -> int:
    """Produce exactly one public stance: +1 (support) or -1 (oppose)."""
    return 1 if rng.random() < support_probability(state) else -1


def _message_count(value: int, name: str) -> int:
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, Integral):
        raise ValueError(f"{name} must be a non-negative integer.")
    if value < 0:
        raise ValueError(f"{name} must be a non-negative integer.")
    return int(value)


def consume_batch(
    state: BetaBelief,
    n_support: int,
    n_oppose: int,
) -> BetaBelief:
    """Homogeneous Beta–Bernoulli update for one consumed batch."""
    n_support = _message_count(n_support, "n_support")
    n_oppose = _message_count(n_oppose, "n_oppose")
    return BetaBelief(state.a + n_support, state.b + n_oppose)


def belief_summary(state: BetaBelief) -> dict[str, float]:
    mean = state.a / (state.a + state.b)
    variance = (
        state.a * state.b
        / ((state.a + state.b) ** 2 * (state.a + state.b + 1.0))
    )
    probabilities = message_probabilities(state)
    return {
        "a": state.a,
        "b": state.b,
        "mean": mean,
        "signed_mean": 2.0 * mean - 1.0,
        "concentration": state.a + state.b,
        "variance": variance,
        "signed_variance": 4.0 * variance,
        "p_support": probabilities["support"],
        "p_oppose": probabilities["oppose"],
    }


## 2. Production probe

The following comparison covers opposing-biased, balanced-propensity, and supporting-biased beliefs. Each diffuse/concentrated pair has the same mean but a different concentration. Balanced propensity is the symmetric boundary, not a center stance; it still generates only $-1$ or $+1$.


In [ ]:
PRODUCTION_STATES = {
    "opposing-biased, diffuse": BetaBelief(2, 5),
    "opposing-biased, concentrated": BetaBelief(20, 50),
    "balanced propensity, diffuse": BetaBelief(2, 2),
    "balanced propensity, concentrated": BetaBelief(20, 20),
    "supporting-biased, diffuse": BetaBelief(5, 2),
    "supporting-biased, concentrated": BetaBelief(50, 20),
}

rng = np.random.default_rng(20260903)
sample_size = 4_000
production_rows = []

for label, state in PRODUCTION_STATES.items():
    summary = belief_summary(state)
    messages = np.array([produce_message(state, rng) for _ in range(sample_size)])
    production_rows.append({
        "belief": label,
        **summary,
        "sampled_support_share": float(np.mean(messages == 1)),
    })

PRODUCTION_RESULTS = pd.DataFrame(production_rows)
display(PRODUCTION_RESULTS[[
    "belief", "a", "b", "signed_mean", "concentration",
    "p_support", "p_oppose", "sampled_support_share",
]].round(4))


## 3. Homogeneous message consumption

If one round contains $n_+$ supportive messages and $n_-$ opposing messages, the baseline posterior is

$$P_{t+1}\sim\operatorname{Beta}(a_t+n_+,b_t+n_-).$$

Message composition determines the direction of posterior movement, while message count determines how much information the batch contributes relative to the prior. This rule treats messages as equal-weight, conditionally independent Bernoulli evidence.


In [ ]:
def transition_record(
    label: str,
    prior: BetaBelief,
    n_support: int,
    n_oppose: int,
) -> dict[str, float | str | int]:
    posterior = consume_batch(prior, n_support, n_oppose)
    before = belief_summary(prior)
    after = belief_summary(posterior)
    total = n_support + n_oppose
    return {
        "batch": label,
        "n_support": n_support,
        "n_oppose": n_oppose,
        "total": total,
        "support_share": n_support / total if total else np.nan,
        "signed_mean_before": before["signed_mean"],
        "signed_mean_after": after["signed_mean"],
        "delta_signed_mean": after["signed_mean"] - before["signed_mean"],
        "concentration_before": before["concentration"],
        "concentration_after": after["concentration"],
        "variance_before": before["variance"],
        "variance_after": after["variance"],
        "p_support_before": before["p_support"],
        "p_support_after": after["p_support"],
        "delta_p_support": after["p_support"] - before["p_support"],
    }


PROBE_PRIOR = BetaBelief(2, 2)


### 3.1 Fixed number, varying stance distribution

Hold the total message count at 10 and vary only the supportive/opposing composition.


In [ ]:
composition_batches = [
    ("0 support / 10 oppose", 0, 10),
    ("2 support / 8 oppose", 2, 8),
    ("5 support / 5 oppose", 5, 5),
    ("8 support / 2 oppose", 8, 2),
    ("10 support / 0 oppose", 10, 0),
]

COMPOSITION_RESULTS = pd.DataFrame([
    transition_record(label, PROBE_PRIOR, n_support, n_oppose)
    for label, n_support, n_oppose in composition_batches
])

display(COMPOSITION_RESULTS[[
    "batch", "support_share", "signed_mean_after", "concentration_after",
    "variance_after", "p_support_after", "delta_p_support",
]].round(4))


### 3.2 Fixed stance distribution, varying message number

Hold the supportive share at 80% and vary only batch size to isolate the effect of message volume.


In [ ]:
volume_batches = [
    ("4 support / 1 oppose", 4, 1),
    ("8 support / 2 oppose", 8, 2),
    ("16 support / 4 oppose", 16, 4),
    ("40 support / 10 oppose", 40, 10),
]

VOLUME_RESULTS = pd.DataFrame([
    transition_record(label, PROBE_PRIOR, n_support, n_oppose)
    for label, n_support, n_oppose in volume_batches
])

display(VOLUME_RESULTS[[
    "batch", "total", "support_share", "signed_mean_after",
    "concentration_after", "variance_after", "p_support_after",
]].round(4))


## 4. One editable transition view

Change only the four inputs below to inspect another prior and message batch. The left panel shows the prior and posterior densities on the signed $[-1,1]$ axis; the right panel compares the two message-production probabilities before and after consumption.


In [ ]:
EXAMPLE_PRIOR = BetaBelief(a=2, b=2)
EXAMPLE_N_SUPPORT = 8
EXAMPLE_N_OPPOSE = 2


def plot_transition(
    prior: BetaBelief,
    n_support: int,
    n_oppose: int,
) -> None:
    posterior = consume_batch(prior, n_support, n_oppose)
    theta = np.linspace(-0.999, 0.999, 600)
    p = (theta + 1.0) / 2.0
    prior_density = beta_distribution.pdf(p, prior.a, prior.b) / 2.0
    posterior_density = beta_distribution.pdf(p, posterior.a, posterior.b) / 2.0

    before = message_probabilities(prior)
    after = message_probabilities(posterior)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    axes[0].plot(theta, prior_density, label=f"prior Beta({prior.a:g}, {prior.b:g})")
    axes[0].plot(
        theta,
        posterior_density,
        label=f"posterior Beta({posterior.a:g}, {posterior.b:g})",
    )
    axes[0].axvline(0.0, color="black", linewidth=0.8, alpha=0.5)
    axes[0].set(xlabel="signed private coordinate θ", ylabel="density")
    axes[0].legend()

    positions = np.arange(2)
    width = 0.35
    axes[1].bar(
        positions - width / 2,
        [before["oppose"], before["support"]],
        width,
        label="before consumption",
    )
    axes[1].bar(
        positions + width / 2,
        [after["oppose"], after["support"]],
        width,
        label="after consumption",
    )
    axes[1].set_xticks(positions, ["oppose (-1)", "support (+1)"])
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set(ylabel="message probability")
    axes[1].legend()

    fig.suptitle(f"Consumed batch: {n_support} support / {n_oppose} oppose")
    plt.show()

    display(pd.DataFrame(
        [belief_summary(prior), belief_summary(posterior)],
        index=["before", "after"],
    ).round(4))


plot_transition(EXAMPLE_PRIOR, EXAMPLE_N_SUPPORT, EXAMPLE_N_OPPOSE)


## 5. Executable verification

These checks cover parameter domains, no-input invariance, positive/negative symmetry, concentration accounting, production-probability bounds, random reproducibility, and basic agreement between analytical and sampled probabilities.


In [ ]:
def run_checks() -> None:
    state = BetaBelief(3, 7)
    assert consume_batch(state, 0, 0) == state

    posterior = consume_batch(state, 8, 2)
    assert posterior.a == state.a + 8
    assert posterior.b == state.b + 2
    assert np.isclose(
        belief_summary(posterior)["concentration"],
        belief_summary(state)["concentration"] + 10,
    )

    for candidate in PRODUCTION_STATES.values():
        probabilities = message_probabilities(candidate)
        assert 0.0 <= probabilities["support"] <= 1.0
        assert 0.0 <= probabilities["oppose"] <= 1.0
        assert np.isclose(probabilities["support"] + probabilities["oppose"], 1.0)

    original = BetaBelief(2, 5)
    mirrored = BetaBelief(5, 2)
    original_after = consume_batch(original, 8, 2)
    mirrored_after = consume_batch(mirrored, 2, 8)
    assert np.isclose(
        belief_summary(original_after)["signed_mean"],
        -belief_summary(mirrored_after)["signed_mean"],
    )
    assert np.isclose(
        support_probability(original_after),
        1.0 - support_probability(mirrored_after),
    )

    base = BetaBelief(4, 4)
    assert support_probability(consume_batch(base, 1, 0)) > support_probability(base)
    assert support_probability(consume_batch(base, 0, 1)) < support_probability(base)

    first_rng = np.random.default_rng(12345)
    second_rng = np.random.default_rng(12345)
    first_messages = [produce_message(base, first_rng) for _ in range(100)]
    second_messages = [produce_message(base, second_rng) for _ in range(100)]
    assert first_messages == second_messages
    assert set(first_messages).issubset({-1, 1})

    sampled_error = (
        PRODUCTION_RESULTS["sampled_support_share"]
        - PRODUCTION_RESULTS["p_support"]
    ).abs()
    assert sampled_error.max() < 0.04

    for invalid_counts in [(-1, 0), (0, -1), (1.5, 0), (0, True)]:
        try:
            consume_batch(base, *invalid_counts)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Invalid counts accepted: {invalid_counts}")

    print("All baseline checks passed.")


run_checks()


## 6. Interpretation boundary

- A symmetric Beta belief represents balanced expression propensity, not a public center stance.
- A single message is a stochastic realization of the private belief; producing a message does not automatically change the producing agent's belief.
- Homogeneous consumption uses raw counts: at the same stance proportion, more messages produce a higher concentration.
- Even when messages conflict, the standard Beta–Bernoulli update interprets more observations as more information; it does not automatically represent confusion.
- Opinion-leader source weights, message quality, credibility, correlation, and selection effects must be added later as separate mechanisms. They cannot be inferred from this baseline's results.
- This notebook provides only isolated software verification and mechanism-behavior checks for formation and updating. It does not support population, network, platform, or empirical claims.
